# 01_identificar_posts_madre — recolección formal

## Objetivo
Construir una query media-anchored para cada combinación de 6 eventos y todos los medios activos declarados en `config/media_accounts.yaml`, ejecutar primero Full-Archive Counts y luego recuperar publicaciones candidatas.

Este notebook no recolecta replies, quotes ni interacciones de usuarios.


## Entradas
- `config/events.yaml`
- `config/media_accounts.yaml`
- `config/search_terms.yaml`
- `.env` con `X_BEARER_TOKEN`

## Salidas
- `outputs/tables/query_counts_formal.csv`
- `data/interim/source_posts_candidates_formal.csv`
- `data/interim/source_posts_candidates_formal_audit.csv`


In [ ]:
import os
import sys
import importlib
from pathlib import Path

import pandas as pd
import yaml
from dotenv import load_dotenv
from IPython.display import display

EXPECTED_EVENT_COUNT = 6
MAX_TERMS = 40
MAX_QUERY_CHARS = int(os.getenv("FORMAL_MAX_QUERY_CHARS", "900"))
COUNTS_GRANULARITY = os.getenv("FORMAL_COUNTS_GRANULARITY", "hour")
MAX_RESULTS_PER_PAGE = int(os.getenv("FORMAL_MAX_RESULTS_PER_PAGE", "100"))
MAX_PAGES_PER_QUERY = int(os.getenv("FORMAL_MAX_PAGES_PER_QUERY", "20"))
TIMEOUT_SECONDS = int(os.getenv("FORMAL_TIMEOUT_SECONDS", "60"))
MAX_RATE_WAIT_SECONDS = int(os.getenv("FORMAL_MAX_RATE_WAIT_SECONDS", "60"))
MAX_429_RETRIES = int(os.getenv("FORMAL_MAX_429_RETRIES", "3"))

# Mantener apagado para evitar repetir llamadas al ejecutar todas las celdas.
RUN_FORMAL_COUNTS = os.getenv("RUN_FORMAL_COUNTS", "false").strip().lower() == "true"
RUN_FORMAL_SEARCH = os.getenv("RUN_FORMAL_SEARCH", "false").strip().lower() == "true"

print("RUN_FORMAL_COUNTS:", RUN_FORMAL_COUNTS)
print("RUN_FORMAL_SEARCH:", RUN_FORMAL_SEARCH)
print("MAX_TERMS:", MAX_TERMS)
print("Selección de medios: todos los medios activos del YAML")


In [ ]:
def is_project_root(path):
    return (path / "config").exists() and (path / "src").exists() and (path / "notebooks").exists()


def find_project_root(start):
    for candidate in [start] + list(start.parents):
        if is_project_root(candidate):
            return candidate
        child = candidate / "HateCR"
        if is_project_root(child):
            return child
    raise FileNotFoundError("No se encontró la raíz del proyecto HateCR")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env", override=False)

import src.x_api as xapi
import src.search_queries as sq
import src.collection as col

importlib.reload(xapi)
importlib.reload(sq)
importlib.reload(col)

CONFIG_DIR = PROJECT_ROOT / "config"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

COUNTS_PATH = OUTPUT_TABLES / "query_counts_formal.csv"
CANDIDATES_PATH = INTERIM_DIR / "source_posts_candidates_formal.csv"
CANDIDATES_AUDIT_PATH = INTERIM_DIR / "source_posts_candidates_formal_audit.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Counts endpoint:", xapi.get_x_api_urls().counts_all_url)
print("Search endpoint:", xapi.get_x_api_urls().search_all_url)


## 1. Carga y validación de configuración formal


In [ ]:
with open(CONFIG_DIR / "events.yaml", "r", encoding="utf-8") as f:
    events_cfg = yaml.safe_load(f) or {}
with open(CONFIG_DIR / "media_accounts.yaml", "r", encoding="utf-8") as f:
    media_cfg = yaml.safe_load(f) or {}
with open(CONFIG_DIR / "search_terms.yaml", "r", encoding="utf-8") as f:
    terms_cfg = yaml.safe_load(f) or {}

all_events = events_cfg.get("events", []) or []
all_media = media_cfg.get("media_accounts", []) or []
term_groups = terms_cfg.get("term_groups", {}) or {}

# Los eventos formales y los medios se controlan desde YAML, no desde listas fijas.
formal_events = [
    e for e in all_events
    if e.get("active", True) and e.get("formal", False)
]
formal_events = sorted(formal_events, key=lambda e: int(e.get("formal_order", 999)))

# Todos los medios activos del YAML forman parte de la recolección ampliada.
formal_media = [m for m in all_media if m.get("active", True)]
missing_handles = [
    str(m.get("media_id"))
    for m in formal_media
    if not str(m.get("x_identity", {}).get("handle", "")).strip()
]
assert not missing_handles, f"Medios activos sin handle de X: {missing_handles}"

media_ids = [str(m.get("media_id")) for m in formal_media]
media_handles = [str(m.get("x_identity", {}).get("handle")).lower() for m in formal_media]
assert len(media_ids) == len(set(media_ids)), "Hay media_id duplicados en media_accounts.yaml"
assert len(media_handles) == len(set(media_handles)), "Hay handles duplicados en media_accounts.yaml"
assert len(formal_events) == EXPECTED_EVENT_COUNT, f"Se esperaban {EXPECTED_EVENT_COUNT} eventos formales y hay {len(formal_events)}."
assert [int(e.get("formal_order")) for e in formal_events] == list(range(EXPECTED_EVENT_COUNT)), "formal_order debe ir de 0 a 5 sin repeticiones."
assert formal_media, "No hay medios activos en media_accounts.yaml"

EXPECTED_QUERY_COUNT = len(formal_events) * len(formal_media)

print("Eventos formales:", len(formal_events))
display(pd.DataFrame(formal_events)[["formal_order", "event_id", "event_name", "event_date"]])
print("Medios activos incluidos:", len(formal_media))
print("Queries esperadas:", EXPECTED_QUERY_COUNT)
display(pd.DataFrame([
    {
        "media_id": m.get("media_id"),
        "media_name": m.get("media_name"),
        "media_handle": m.get("x_identity", {}).get("handle"),
        "media_type": m.get("media_type"),
    }
    for m in formal_media
]))


## 2. Construcción de queries formales
Se construye una query por combinación evento-medio. Cada query contiene `from:{media_handle}`, hasta 40 términos unidos con `OR`, `lang:es` y `-is:retweet`.


In [ ]:
query_specs = sq.build_formal_event_media_query_specs(
    events=formal_events,
    media_accounts=formal_media,
    term_groups=term_groups,
    max_terms=MAX_TERMS,
    max_query_chars=MAX_QUERY_CHARS,
    collection_scope="media_anchored",
)
query_specs_df = pd.DataFrame(query_specs)

assert len(query_specs_df) == EXPECTED_QUERY_COUNT, f"Se esperaban {EXPECTED_QUERY_COUNT} queries y se construyeron {len(query_specs_df)}"
assert query_specs_df["event_id"].nunique() == EXPECTED_EVENT_COUNT
assert query_specs_df["media_id"].nunique() == len(formal_media)
assert query_specs_df["term_count"].between(1, MAX_TERMS).all()
assert query_specs_df["query"].str.contains(" OR ", regex=False).all()
assert query_specs_df.apply(
    lambda row: f"from:{str(row['media_handle']).lower()}" in str(row["query"]).lower(),
    axis=1,
).all()

pairs = query_specs_df.groupby(["event_id", "media_id"]).size()
assert (pairs == 1).all(), "Debe existir una sola query por evento y medio."

print("Queries construidas:", len(query_specs_df))
display(query_specs_df[[
    "formal_order", "event_id", "media_id", "media_handle", "term_count",
    "start_time", "end_time", "query"
]])


## 3. Full-Archive Counts
Esta fase debe ejecutarse antes de buscar candidatos. Para repetirla, define `RUN_FORMAL_COUNTS=true` antes de iniciar el kernel.


In [ ]:
headers = None
urls = xapi.get_x_api_urls()

if RUN_FORMAL_COUNTS:
    xapi.validate_x_api_configuration()
    headers = xapi.build_x_auth_headers()
    counts_df = col.collect_full_archive_query_counts(
        query_specs=query_specs,
        headers=headers,
        urls=urls,
        granularity=COUNTS_GRANULARITY,
        timeout_seconds=TIMEOUT_SECONDS,
        max_rate_limit_wait_seconds=MAX_RATE_WAIT_SECONDS,
        max_429_retries=MAX_429_RETRIES,
    )
    counts_df.to_csv(COUNTS_PATH, index=False)
    print("[OK] Guardado:", COUNTS_PATH)
elif COUNTS_PATH.exists():
    loaded_counts_df = pd.read_csv(COUNTS_PATH)
    expected_keys = {(str(s["event_id"]), str(s["media_id"])) for s in query_specs}
    loaded_keys = {
        (str(row.event_id), str(row.media_id))
        for row in loaded_counts_df[["event_id", "media_id"]].drop_duplicates().itertuples(index=False)
    }
    if loaded_keys == expected_keys:
        counts_df = loaded_counts_df
        print("[INFO] Counts existentes cargados:", COUNTS_PATH)
    else:
        counts_df = pd.DataFrame()
        print(
            "[WARN] El archivo de Counts es de otra selección de medios/eventos "
            f"({len(loaded_keys)} de {len(expected_keys)} pares). Ejecuta con RUN_FORMAL_COUNTS=true."
        )
else:
    counts_df = pd.DataFrame()
    print("[SKIP] Counts no ejecutados. Activa RUN_FORMAL_COUNTS=true.")

if not counts_df.empty:
    display(counts_df)


## 4. Búsqueda Full Archive de posts candidatos
Se ejecuta solamente después de Counts. Para repetirla, define `RUN_FORMAL_SEARCH=true`.


In [ ]:
if RUN_FORMAL_SEARCH:
    if counts_df.empty:
        raise RuntimeError("Primero debes ejecutar o cargar Full-Archive Counts.")
    if headers is None:
        headers = xapi.build_x_auth_headers()

    count_keys = counts_df[["event_id", "media_id", "status", "total_tweet_count"]].copy()
    count_keys["total_tweet_count"] = pd.to_numeric(count_keys["total_tweet_count"], errors="coerce").fillna(0)
    count_lookup = {
        (str(row.event_id), str(row.media_id)): (str(row.status), float(row.total_tweet_count))
        for row in count_keys.itertuples(index=False)
    }

    search_specs = []
    skipped_zero_specs = []
    for spec in query_specs:
        status, total = count_lookup.get((str(spec["event_id"]), str(spec["media_id"])), ("missing", -1))
        if status == "ok" and total == 0:
            skipped_zero_specs.append(spec)
        else:
            search_specs.append(spec)

    print("Queries con conteo positivo o desconocido:", len(search_specs))
    print("Queries omitidas por conteo cero:", len(skipped_zero_specs))

    candidates_df, search_audit_df = col.collect_source_post_candidates(
        query_specs=search_specs,
        headers=headers,
        urls=urls,
        use_full_archive=True,
        max_results_per_page=MAX_RESULTS_PER_PAGE,
        max_pages_per_query=MAX_PAGES_PER_QUERY,
        timeout_seconds=TIMEOUT_SECONDS,
        sleep_seconds=0.2,
        max_rate_limit_wait_seconds=MAX_RATE_WAIT_SECONDS,
        max_429_retries=MAX_429_RETRIES,
        recent_days=7,
        allow_recent_fallback=False,
        collection_scope="media_anchored",
        enable_media_source_posts=True,
        dedupe_columns=["event_id", "media_id", "id"],
    )

    skipped_zero_audit_df = pd.DataFrame([
        {
            "event_id": spec.get("event_id"),
            "media_id": spec.get("media_id"),
            "media_handle": spec.get("media_handle"),
            "search_mode": spec.get("search_mode"),
            "term_batch_id": spec.get("term_batch_id"),
            "term_count": spec.get("term_count"),
            "term_group_names": spec.get("term_group_names"),
            "query": spec.get("query"),
            "start_time": spec.get("start_time"),
            "end_time": spec.get("end_time"),
            "endpoint_attempted": "none",
            "endpoint_used": "none",
            "status": "skipped_zero_count",
            "status_code": None,
            "n_rows": 0,
            "error_summary": "Full-Archive Counts returned zero",
        }
        for spec in skipped_zero_specs
    ])
    candidates_audit_df = pd.concat([search_audit_df, skipped_zero_audit_df], ignore_index=True, sort=False)

    if candidates_df.empty:
        candidates_df["formal_event_count"] = pd.Series(dtype="int64")
        candidates_df["duplicate_across_formal_events"] = pd.Series(dtype="bool")
        candidates_df["formal_event_memberships"] = pd.Series(dtype="object")
    else:
        event_count_by_tweet = candidates_df.groupby("tweet_id")["event_id"].transform("nunique")
        candidates_df["formal_event_count"] = event_count_by_tweet
        candidates_df["duplicate_across_formal_events"] = event_count_by_tweet > 1
        event_memberships = (
            candidates_df.groupby("tweet_id")["event_id"]
            .agg(lambda values: "|".join(sorted(set(values.astype(str)))))
        )
        candidates_df["formal_event_memberships"] = candidates_df["tweet_id"].map(event_memberships)

    candidates_df.to_csv(CANDIDATES_PATH, index=False)
    candidates_audit_df.to_csv(CANDIDATES_AUDIT_PATH, index=False)
    print("[OK] Guardado:", CANDIDATES_PATH)
    print("[OK] Auditoría:", CANDIDATES_AUDIT_PATH)
else:
    candidates_df = pd.read_csv(CANDIDATES_PATH) if CANDIDATES_PATH.exists() else pd.DataFrame()
    candidates_audit_df = pd.read_csv(CANDIDATES_AUDIT_PATH) if CANDIDATES_AUDIT_PATH.exists() else pd.DataFrame()
    print("[SKIP] Búsqueda no ejecutada. Activa RUN_FORMAL_SEARCH=true.")


## 5. Validaciones formales


In [ ]:
print("1) Queries construidas")
print("Total:", len(query_specs_df))
display(query_specs_df.groupby("event_id").size().reset_index(name="n_queries"))
display(query_specs_df.groupby("media_id").size().reset_index(name="n_queries"))

print("2) Conteos por evento")
if not counts_df.empty:
    display(
        counts_df.groupby(["formal_order", "event_id"], dropna=False)["total_tweet_count"]
        .sum().reset_index().sort_values("formal_order")
    )

print("3) Conteos por medio")
if not counts_df.empty:
    display(
        counts_df.groupby(["media_id", "media_handle"], dropna=False)["total_tweet_count"]
        .sum().reset_index().sort_values("total_tweet_count", ascending=False)
    )

print("4) Posts candidatos por evento y medio")
if candidates_df.empty:
    print("Sin candidatos cargados todavía.")
else:
    display(
        candidates_df.groupby(["event_id", "media_id", "media_handle"], dropna=False)
        .size().reset_index(name="n_candidates")
        .sort_values(["event_id", "media_id"])
    )

print("5) Posibles duplicados")
if candidates_df.empty:
    print("Sin candidatos para validar duplicados.")
else:
    id_col = "tweet_id" if "tweet_id" in candidates_df.columns else "id"
    duplicates_by_id = candidates_df[candidates_df.duplicated(subset=[id_col], keep=False)].copy()
    print("Filas duplicadas por tweet_id:", len(duplicates_by_id))

    text_col = "source_post_text" if "source_post_text" in candidates_df.columns else "text"
    duplicates_by_text = candidates_df[
        candidates_df[text_col].fillna("").ne("")
        & candidates_df.duplicated(subset=["media_id", text_col], keep=False)
    ].copy()
    print("Posibles duplicados por medio + texto:", len(duplicates_by_text))
    display(duplicates_by_id.head(20))
    display(duplicates_by_text.head(20))

print("6) Auditoría de endpoints")
if not candidates_audit_df.empty:
    display(
        candidates_audit_df.groupby(["endpoint_used", "status", "status_code"], dropna=False)
        .size().reset_index(name="n_queries")
    )


## Cierre metodológico
La unidad recolectada aquí es el post madre publicado por uno de los medios activos configurados en `media_accounts.yaml`. No se recolectan respuestas en este notebook. El acceso es exclusivamente mediante la API oficial de X y Full Archive; no se usa scraping, Selenium, proxies ni evasión de bloqueos.
